In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso)

from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

In [14]:
mpg_df = pd.read_csv('car-mpg.csv')
mpg_df.head()

,mpg,cyl,disp,hp,wt,acc,yr,origin,car_type,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,0,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,0,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,0,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,0,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,0,ford torino


In [15]:
mpg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   mpg       398 non-null    float64
 1   cyl       398 non-null    int64  
 2   disp      398 non-null    float64
 3   hp        398 non-null    object 
 4   wt        398 non-null    int64  
 5   acc       398 non-null    float64
 6   yr        398 non-null    int64  
 7   origin    398 non-null    int64  
 8   car_type  398 non-null    int64  
 9   car_name  398 non-null    object 
dtypes: float64(3), int64(5), object(2)
memory usage: 31.2+ KB


In [16]:
mpg_df.drop('car_name', axis=1, inplace=True)

In [17]:
mpg_df['origin'] = mpg_df['origin'].replace({1:'america',
                                             2:'europe',
                                             3:'asia'})
mpg_df = pd.get_dummies(mpg_df, columns=['origin'])

In [19]:
mpg_df = mpg_df.replace('?', np.nan)

In [21]:
mpg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mpg             398 non-null    float64
 1   cyl             398 non-null    int64  
 2   disp            398 non-null    float64
 3   hp              392 non-null    object 
 4   wt              398 non-null    int64  
 5   acc             398 non-null    float64
 6   yr              398 non-null    int64  
 7   car_type        398 non-null    int64  
 8   origin_america  398 non-null    bool   
 9   origin_asia     398 non-null    bool   
 10  origin_europe   398 non-null    bool   
dtypes: bool(3), float64(3), int64(4), object(1)
memory usage: 26.2+ KB


In [31]:
# change the hp column to numeric in order to cleanup NaN values
mpg_df['hp']=pd.to_numeric(mpg_df['hp'])

In [32]:
mpg_df['hp']=mpg_df['hp'].fillna(mpg_df['hp'].mean())

In [34]:
mpg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mpg             398 non-null    float64
 1   cyl             398 non-null    int64  
 2   disp            398 non-null    float64
 3   hp              398 non-null    float64
 4   wt              398 non-null    int64  
 5   acc             398 non-null    float64
 6   yr              398 non-null    int64  
 7   car_type        398 non-null    int64  
 8   origin_america  398 non-null    bool   
 9   origin_asia     398 non-null    bool   
 10  origin_europe   398 non-null    bool   
dtypes: bool(3), float64(4), int64(4)
memory usage: 26.2 KB


### Split the data

In [36]:
X = mpg_df.drop('mpg', axis=1)
y = mpg_df['mpg']

In [37]:
from sklearn import preprocessing

# scale all the columns of the mpg_df. This will produce a numpy array
X_scaled = preprocessing.scale(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
(X_train, X_test,
 y_train, y_test) = train_test_split(X_scaled,
                                     y,
                                     test_size=0.3,
                                     random_state=1)

## Linear regression model

In [42]:
regression_model = LinearRegression().fit(X_train,y_train)

In [50]:
print('Coefficients:\n')
for idx, col_name in enumerate(X_train.columns):
    print('{} : {}'.format(col_name,round(regression_model.coef_[idx],4)))

Coefficients:

cyl : 2.4882
disp : 2.5179
hp : -1.7179
wt : -5.5908
acc : 0.1383
yr : 2.9379
car_type : 2.9658
origin_america : -0.5754
origin_asia : 0.3393
origin_europe : 0.3759


In [53]:
intercept = regression_model.intercept_
print("The intercept for our model is {}".format(intercept))

The intercept for our model is 23.665938912014894


## Create a regularized RIDGE model and note the coefficients

In [57]:
ridge = Ridge(alpha=0.3).fit(X_train, y_train)

In [58]:
print('Ridge Model Coefficients:\n')
for idx, col_name in enumerate(X_train.columns):
    print('{} : {}'.format(col_name,round(ridge.coef_[idx],4)))

Ridge Model Coefficients:

cyl : 2.4533
disp : 2.4271
hp : -1.7159
wt : -5.5109
acc : 0.1242
yr : 2.9292
car_type : 2.9334
origin_america : -0.5716
origin_asia : 0.3386
origin_europe : 0.3719


## Create a regularized LASSO model and note the coefficients

In [59]:
lasso = Lasso(alpha=0.2).fit(X_train, y_train)

In [60]:
print('Lasso Model Coefficients:\n')
for idx, col_name in enumerate(X_train.columns):
    print('{} : {}'.format(col_name,round(lasso.coef_[idx],4)))

Lasso Model Coefficients:

cyl : 0.0
disp : 0.0
hp : -0.354
wt : -4.0069
acc : 0.0
yr : 2.6419
car_type : 1.0725
origin_america : -0.5457
origin_asia : 0.0
origin_europe : 0.0


## Compare the regression scores

In [61]:
# Regression model scores
print(regression_model.score(X_train, y_train))
print(regression_model.score(X_test, y_test))

0.8342401219581603
0.8511803647501518


In [62]:
# Ridge model scores
print(ridge.score(X_train, y_train))
print(ridge.score(X_test, y_test))

0.834224870068746
0.8517212637940696


In [63]:
# Lasso model scores
print(lasso.score(X_train, y_train))
print(lasso.score(X_test, y_test))

0.8114684694905746
0.8546438954465446


## Polynomial models

In [65]:
from sklearn.preprocessing import PolynomialFeatures

In [66]:
poly = PolynomialFeatures(degree=2, interaction_only=True)

In [67]:
X_poly = poly.fit_transform(X_scaled)
(X_train, X_test,
 y_train, y_test) = train_test_split(X_poly,
                                     y,
                                     test_size=0.3,
                                     random_state=1)
X_train.shape

(278, 56)

In [72]:
regression_model.fit(X_train, y_train)
print(regression_model.coef_)

[ 2.63182252e-13  1.31599440e+10 -4.48715063e+00 -1.93019126e+00
 -3.08500239e+00 -1.43209022e+00  3.01452419e+00  1.75221952e+10
  6.18040696e+09 -4.21789403e+10  3.37570918e+10 -1.70446993e+00
 -1.08533050e+00 -9.46687603e-02  2.75273460e+00 -2.00529103e+00
  4.21623085e+10  7.48996825e+10  2.80345204e+10  2.67590189e+10
  4.08721798e-01  5.03338826e-01 -5.53701460e-01  3.45233819e+00
 -2.39987131e+00  1.01384635e+10  8.35580356e+09  7.97563511e+09
 -7.65199362e-03 -5.58018301e-01 -1.82581630e+00 -6.16032423e-01
 -1.57685004e+10 -1.29959033e+10 -1.24046218e+10 -2.01332092e-01
  5.36966324e-01 -3.14018822e+00 -1.87534734e+10 -1.54560244e+10
 -1.47528134e+10  5.44952393e-01  1.65619469e+00 -3.57744193e+09
 -2.94841541e+09 -2.81426978e+09  3.61122131e-01  1.46640520e+09
  1.20856516e+09  1.15357843e+09  1.12454434e+10 -1.05298473e+10
 -1.00507652e+10 -2.19234193e+08  5.84346651e+10 -3.16357260e+10]


In [75]:
print(regression_model.score(X_train, y_train))
print(regression_model.score(X_test, y_test))

0.9146579450381869
0.8596141508588334


In [73]:
ridge = Ridge(alpha=.3)
ridge.fit(X_train,y_train)
print ("Ridge model:", (ridge.coef_))

Ridge model: [ 0.          3.77317587 -2.98001611 -1.88899735 -3.68041028 -1.19486823
  3.01386595  2.10315232  0.08787373  0.19881924 -0.31999948 -1.64346514
 -1.10962894  0.01086857  2.43224747 -1.70465304  0.95204418 -2.29389943
  4.68797625 -1.99547658  0.35351176  0.58928008 -0.41707484  3.02832783
 -2.36004236  1.79760152 -2.60575716  0.44488965 -0.01930968 -0.48184163
 -1.81894573 -0.78441907 -0.91298298  0.41681246  0.72388487 -0.26609462
  0.60324441 -2.7609145  -0.26485084  0.61222753 -0.30473717  0.51502548
  1.39932713 -0.4484515  -0.24020151  0.82171334  0.357644   -0.47916303
  0.35369255  0.23855045 -1.41338822  2.00194244 -0.30069742 -0.30074861
  0.37307684  0.06739745]


In [74]:
print(ridge.score(X_train, y_train))
print(ridge.score(X_test, y_test))

0.913946541194562
0.8619635470076642


In [76]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train,y_train)
print ("Lasso model:", (lasso.coef_))

Lasso model: [ 0.         -0.         -0.         -1.54039148 -5.26220009 -0.
  2.8770539   0.0461137  -0.09349614  0.          0.         -0.
 -0.          0.          0.29278798 -0.          0.         -0.
  0.12418985 -0.          0.          1.20282992  0.          0.
 -0.          0.          0.         -0.          0.          0.
 -0.68937812 -0.          0.         -0.         -0.         -0.
 -0.         -0.6644578   0.         -0.         -0.          0.17189654
 -0.         -0.61547892  0.          0.36609144  0.         -0.37601834
  0.          0.         -0.         -0.          0.18367341 -0.
 -0.         -0.        ]


In [77]:
print(lasso.score(X_train, y_train))
print(lasso.score(X_test, y_test))

0.8899826115970141
0.8797709732777949
